In [2]:
# ipca_pipeline.py (or paste into a notebook cell)

from __future__ import annotations
import os
import contextlib
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict, Any

import numpy as np
import pandas as pd
from ipca import InstrumentedPCA

def run_pipeline(
    start_yyyymm: str = "199001",
    end_yyyymm: str = "202412",
    max_nan_frac: float = 0.6,
    cutoff: str = "2015-12-31",
    n_factors: int = 3,
) -> Dict[str, Any]:
    df_raw = download_openap_data(start_yyyymm, end_yyyymm)
    df_keep, dropped = remove_mostly_nan_columns(df_raw, max_nan_frac=max_nan_frac)
    df_filled = fill_remaining_missing(df_keep)

    panel = build_ipca_panel(df_filled, ret_col="excess_ret", shift_target=True)
    train_df, test_df = split_train_test_by_date(panel.df, cutoff=cutoff)

    X_train = train_df[panel.char_cols].to_numpy(np.float64)
    y_train = train_df["y_ipca"].to_numpy(np.float64)
    idx_train = train_df[["i_idx", "t_idx"]].to_numpy(np.int64)

    X_test = test_df[panel.char_cols].to_numpy(np.float64)
    y_test = test_df["y_ipca"].to_numpy(np.float64)
    idx_test = test_df[["i_idx", "t_idx"]].to_numpy(np.int64)

    mod = fit_ipca(X_train, y_train, idx_train, n_factors=n_factors, silent=True)
    r2_train = score_ipca(mod, X_train, y_train, idx_train, mean_factor=False)
    r2_test = score_ipca(mod, X_test, y_test, idx_test, mean_factor=True)  # unseen dates

    return {
        "model": mod,
        "dropped_cols": dropped,
        "char_cols": panel.char_cols,
        "r2_train": r2_train,
        "r2_test_mean_factor": r2_test,
        "train_shape": X_train.shape,
        "test_shape": X_test.shape,
    }


In [3]:
%cd ..
import importlib
import src.ipca_workflow as IPCAWorkflow

importlib.reload(IPCAWorkflow)

/Users/apple/Desktop/Academics/Research_work/TheVirtueOfComplexity_PaperReplication_Experimentation-master


<module 'src.ipca_workflow' from '/Users/apple/Desktop/Academics/Research_work/TheVirtueOfComplexity_PaperReplication_Experimentation-master/src/ipca_workflow.py'>

In [4]:
new_mod = IPCAWorkflow.IPCAWorkflow()

In [6]:
char_data = new_mod.download_openap_data(start_yyyymm="198001", end_yyyymm="202601")

WRDS recommends setting up a .pgpass file.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done

Data is downloaded: 11 mins


In [7]:
returns_data = new_mod.download_crsp_returns_wrds(start_date="1980-01-01", end_date="2025-12-31")

WRDS recommends setting up a .pgpass file.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [8]:
char_data, dropped_cols = new_mod.remove_mostly_nan_columns(char_data, max_nan_frac=0.5)
print(f"Dropped columns: {dropped_cols}")

Dropped columns: ['AOP', 'AccrualsBM', 'Activism1', 'Activism2', 'AdExp', 'AgeIPO', 'AnalystRevision', 'AnalystValue', 'BetaTailRisk', 'BrandInvest', 'CBOperProf', 'CPVolSpread', 'Cash', 'ChAssetTurnover', 'ChForecastAccrual', 'ChNAnalyst', 'ChangeInRecommendation', 'CitationsRD', 'CompEquIss', 'CompositeDebtIssuance', 'ConsRecomm', 'CustomerMomentum', 'DelBreadth', 'DelDRC', 'DivSeason', 'DivYieldST', 'DownRecomm', 'EP', 'EarnSupBig', 'EarningsConsistency', 'EarningsForecastDisparity', 'EarningsStreak', 'EarningsSurprise', 'EntMult', 'ExclExp', 'FEPS', 'FR', 'FirmAgeMom', 'ForecastDispersion', 'Frontier', 'Governance', 'GrAdExp', 'GrSaleToGrInv', 'HerfAsset', 'HerfBE', 'IO_ShortInterest', 'IndRetBig', 'IntanBM', 'IntanCFP', 'IntanEP', 'IntanSP', 'InvGrowth', 'Investment', 'MS', 'MeanRankRevGrowth', 'Mom6mJunk', 'MomOffSeason06YrPlus', 'MomOffSeason11YrPlus', 'MomOffSeason16YrPlus', 'MomRev', 'MomSeason06YrPlus', 'MomSeason11YrPlus', 'MomSeason16YrPlus', 'MomVol', 'NetDebtPrice', 'NetP

In [9]:
char_data, kept_chars, dropped_chars = new_mod.drop_low_std_and_high_corr(
    char_data,
    min_std=1e-6,   # tune as needed
    max_corr=0.98,  # tune as needed
)

/usr/local/lib/python3.11/site-packages/numpy/core/_methods.py:49: RuntimeWarning: invalid value encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
/usr/local/lib/python3.11/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


In [10]:
char_data = new_mod.fill_remaining_missing(char_data)

/usr/local/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


In [11]:
returns_data = returns_data.rename(columns={"excess_ret": "ret_adj"})
char_data = new_mod.merge_openap_with_crsp_returns(char_data, returns_data)


In [12]:
char_data["y_ipca"] = char_data["excess_ret"].shift(-1)  # next period's return as target

In [63]:
pred_df = new_mod.rolling_ipca_predictions(
    char_data=char_data,          # or char_data directly
    forecast_start="2016-01-01",
    target_col="y_ipca",            # auto-built from excess_ret if missing
    n_factors=3,
    train_window_months=120,        # rolling 10y window; set None for expanding
    min_train_obs=5000,
    normalize=True,
    mean_factor=True,
    silent=True,
)
pred_df.head()

In [ ]:
import os
import importlib
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
import src.ipca_workflow as ipw

importlib.reload(ipw)

factor_grid = range(3,16)  # choose your factor models
max_workers = min(len(factor_grid), max(1, (os.cpu_count() or 2) // 2))

def run_one_factor(k: int):
    wf = ipw.IPCAWorkflow()
    pred = wf.rolling_ipca_predictions(
        char_data=char_data,          # or filtered char_data
        forecast_start="2016-01-01",
        n_factors=k,
        train_window_months=120,      # or None
        normalize=True,
        silent=True,
        iter_tol=1e-3,
        max_iter=500,
    )
    pred["n_factors"] = k
    return k, pred

results = {}
with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = {ex.submit(run_one_factor, k): k for k in factor_grid}
    for fut in as_completed(futures):
        k, pred_df = fut.result()
        results[k] = pred_df

# combined predictions for all factor models
all_preds = pd.concat(results.values(), ignore_index=True).sort_values(["n_factors", "yyyymm", "permno"])
all_preds.head()


In [ ]:
results